<a href="https://colab.research.google.com/github/Aswanth0704/gpu-programming-cpp/blob/main/0_Execution_Space_Memory_Space.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Which is faster GPU or CPU? <br>
We need additional context to answer this. Copying a single byte of data is five times faster on a CPU than on a GPU but when it comes to copying gigabytes of data, then GPU can do it faster than the CPU.

# Execution Spaces

In [1]:
import os
if os.getenv("COLAB_RELEASE_TAG"): # need this for running on GPU
  !mkdir -p Sources
  !wget https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.02-Execution-Spaces/Sources/ach.h -nv -O Sources/ach.h

2026-09-03 19:24:46 URL:https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.02-Execution-Spaces/Sources/ach.h [2893/2893] -> "Sources/ach.h" [1]


Solve a simple heat transfer equation on a CPU

In [2]:
%%writefile Sources/cpu-cooling.cpp
#include <cstdio>
#include <vector>

int main(){
  float k = 0.5;
  float ambient_temp = 20;
  std::vector<float> temp {42, 24, 50};

  std::printf("step   temp[0]   temp[1]   temp[2]\n");

  for (int step = 0; step < 3; step++){

    for (int i = 0; i< temp.size(); i++){
      float float_diff = ambient_temp - temp[i];
      temp[i] = temp[i] + k*float_diff; // transform each element in the vector
    }

    std::printf("%d     %.2f      %.2f    %.2f\n", step, temp[0], temp[1], temp[2]);
  }

}

Writing Sources/cpu-cooling.cpp


In [3]:
!g++ Sources/cpu-cooling.cpp -o /tmp/a.out # compile the code
!/tmp/a.out # run the executable

step   temp[0]   temp[1]   temp[2]
0     31.00      22.00    35.00
1     25.50      21.00    27.50
2     22.75      20.50    23.75


- g++ compiler consumed C++ code and produced an executable file a.out which is a set of machine instructions.
- Differnet CPUs support different sets of instructions. Eg. x86 compiles to vfmadd132ss instruction, ARM architecture requires vmla.f32 instruction.
- Similarly, GPUs need their own instruction and compiler. NVIDIA cuda compiler (NVCC)

In [4]:
# let's use NVCC compiler
!nvcc -x cu -arch=native Sources/cpu-cooling.cpp -o /tmp/a.out # compile the code
!/tmp/a.out # run the executable

step   temp[0]   temp[1]   temp[2]
0     31.00      22.00    35.00
1     25.50      21.00    27.50
2     22.75      20.50    23.75


Great but none of the above code runs on the GPU yet. Thats because GPUs allow for heterogenous programming

# Heterogenus Programing Model
- GPUs are accelerators rather than standalone processors. A lot of computational work for example network and file system are still handled by CPUs.
- CUDA program always starts on CPU. We need to explicitly specify which code runs on GPU. This is called **execution space**
- Execution space is partitioned into host (CPU) and device (GPU). By default code runs on CPU (host side).
- CUDA compiler, NVCC is accompained by core libraries like Thrust for example, which help us specify where to run a given algorithm.

In [5]:
%%writefile Sources/gpu-cooling.cpp

#include <algorithm>
#include <cstdio>
#include <vector>

int main(){
  float k = 0.5;
  float ambient_temp = 20;
  std::vector<float> temp{42, 24, 50};

  auto transformation = [=] (float temp){
    return temp + k*(ambient_temp - temp);
  };

  std::printf("step   temp[0]   temp[1]   temp[2]\n");

  for (int step = 0; step < 3; step++){
    std::transform(temp.begin(), temp.end(), temp.begin(), transformation);
    std::printf("%d     %.2f      %.2f    %.2f\n", step, temp[0], temp[1], temp[2]);
  }

}

Writing Sources/gpu-cooling.cpp


In [7]:
!nvcc Sources/gpu-cooling.cpp -x cu -arch=native -o /tmp/a.out
!/tmp/a.out

step   temp[0]   temp[1]   temp[2]
0     31.00      22.00    35.00
1     25.50      21.00    27.50
2     22.75      20.50    23.75


- Thrust library provides standard aglorithms and containers that run on GPU.

In [10]:
%%writefile Sources/thrust-cooling.cpp

#include <thrust/execution_policy.h>
#include <thrust/universal_vector.h>
#include <thrust/transform.h>

int main(){
  float k = 0.5;
  float ambient_temp = 20;
  thrust::universal_vector<float> temp{42, 24, 50};
  // the below function can be excuted from both host (CPU) and device (GPU)
  auto transformation = [=]__host__ __device__ (float temp){
    return temp + k*(ambient_temp - temp);
  };

  std::printf("step   temp[0]   temp[1]   temp[2]\n");
  for(int step = 0; step < 3; step++){
    thrust::transform(thrust::device, temp.begin(), temp.end(), temp.begin(), transformation);
    std::printf("%d     %.2f      %.2f    %.2f\n", step, temp[0], temp[1], temp[2]);
  }
}

Overwriting Sources/thrust-cooling.cpp


In [11]:
!nvcc --extended-lambda Sources/thrust-cooling.cpp -x cu -arch=native -o /tmp/a.out
!/tmp/a.out

step   temp[0]   temp[1]   temp[2]
0     31.00      22.00    35.00
1     25.50      21.00    27.50
2     22.75      20.50    23.75
